## Creating audio model (USE PYTHON VERSION 3.10.20)
### Marissa

### Step 1: Define a function that extracts MFCC features

In [6]:
import librosa
import numpy as np

X, y = [], []

def extract_mfcc(audio_path):

    audio, sample_rate = librosa.load(audio_path, res_type = "kaiser_fast")
    
    mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    mfcc = np.mean(mfcc.T, axis=0)

    return mfcc

### Step 2: Iterate over each audio clip in each genre to extract MFCC features

In [7]:
# Audio files
import pandas as pd
import os

DATASET_PATH = "data/genres_original" # Will need to alter based on your own dataset path.


for genre in os.listdir(DATASET_PATH):
    genre_path = os.path.join(DATASET_PATH, genre)

    if not os.path.isdir(genre_path):
        continue

    for file in os.listdir(genre_path):
        if file.endswith(".wav"):
            file_path = os.path.join(genre_path, file)

            try:
                mfcc = extract_mfcc(file_path)
                X.append(mfcc)
                y.append(genre)
            except Exception as e: # Removes corrupted files (should only be 1)
                print("Skipping:", file_path, e)

print("Loaded", len(X), "files")

/var/folders/k2/xkhh4x_x2999dprlhrz1h_r00000gn/T/ipykernel_24225/1885886271.py:8: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sample_rate = librosa.load(audio_path, res_type = "kaiser_fast")
/opt/miniconda3/envs/ml-III-project/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Skipping: data/genres_original/jazz/jazz.00054.wav 
Loaded 999 files


### Step 3: Convert to dataframe (rows are audio clips, columns are MFCC features)

In [8]:
X = np.array(X)
y = np.array(y)

df = pd.DataFrame(X)
df["genre"] = y

### Step 4: Check shape of X and y

In [9]:
# X should have shape (999, mfcc_value) and y should have shape (999,)
print(X.shape)
print(y.shape)

(999, 40)
(999,)


### Step 5: One-hot encode genre labels

In [11]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

y_categorical = to_categorical(y_encoded)

### Step 6: Train-test split

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42
)

### Step 7: Standard Scaling

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Step 8: Building MLP neural network

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential()

model.add(Dense(256, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.3))

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))

model.add(Dense(y_train.shape[1], activation='softmax'))

/opt/miniconda3/envs/ml-III-project/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Step 9: Compile Model

In [16]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### Step 10: Import Early Stopping and Train on Data

In [21]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9797 - loss: 0.0478 - val_accuracy: 0.6313 - val_loss: 2.0152
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9890 - loss: 0.0351 - val_accuracy: 0.6500 - val_loss: 2.0460
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9890 - loss: 0.0363 - val_accuracy: 0.6250 - val_loss: 2.1180
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9765 - loss: 0.0516 - val_accuracy: 0.6375 - val_loss: 2.0828
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9797 - loss: 0.0571 - val_accuracy: 0.6500 - val_loss: 2.0273
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9828 - loss: 0.0473 - val_accuracy: 0.6375 - val_loss: 2.0210
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9828 - loss: 0.0491 - val_accuracy: 0.6500 - val_loss: 1.9975
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9890 - loss: 0.0421 - val_accuracy: 0.6625 - v

### Step 11: Calculate Model Accuracy

In [22]:
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6700 - loss: 2.1661 
Test Accuracy: 0.6700000166893005
